# Modelo de Fair Value EUR/CHF

**Objetivo:** Estimar el valor teórico ('fair value') del tipo de cambio EUR/CHF diario usando:
- **Diferencial de tipos a corto plazo**: rendimiento del bono alemán 1Y (proxy EUR) menos bono suizo 1Y (proxy CHF)
- **Proxy de safe haven**: VIX (volatilidad implícita del S&P 500)

**Marco teórico:**
- Un diferencial de tipos mayor (DE > CH) → flujos hacia EUR → EUR/CHF ↑
- Mayor aversión al riesgo (VIX ↑) → demanda de CHF como refugio → EUR/CHF ↓

**Periodo:** desde enero 2015 (post-abandono del peg SNB a 1.20)

**Metodología:**
1. Test de raíces unitarias (ADF)
2. Test de cointegración (Engle-Granger)
3. Regresión OLS de largo plazo (fair value)
4. Modelo de Corrección de Error (ECM) para dinámica de corto plazo
5. Señales de sobre/infravaloración (±1σ, ±2σ)

In [ ]:
# ─── Instalación de dependencias (ejecutar si es necesario) ───
# !pip install yfinance pandas_datareader statsmodels matplotlib seaborn

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import yfinance as yf
import pandas_datareader.data as web

import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller, coint
from statsmodels.stats.diagnostic import acorr_ljungbox

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.gridspec as gridspec
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'figure.dpi': 120, 'font.size': 10})
pd.set_option('display.float_format', '{:.4f}'.format)

print('Librerías cargadas correctamente.')

## 1. Configuración

In [ ]:
# ─── Parámetros globales ───────────────────────────────────────────────────────
START = '2015-01-01'
END   = pd.Timestamp.today().strftime('%Y-%m-%d')

# Tickers yfinance
YF_TICKERS = {
    'eurchf': 'EURCHF=X',   # Tipo de cambio EUR/CHF
    'vix'   : '^VIX',       # Índice de volatilidad CBOE
}

# Tickers stooq (rendimientos de bonos soberanos a 1 año)
# Fuente: stooq.com vía pandas_datareader
# Alternativa FRED: 'EUR3MTD156N' (Euribor 3M) y 'CHF3MTD156N' (LIBOR CHF 3M)
STOOQ_TICKERS = {
    'de1y': '1YDEY.B',    # Bono alemán 1 año
    'ch1y': '1YCHY.B',    # Bono suizo 1 año (Eidgenossenschaft)
}

print(f'Periodo: {START} → {END}')

## 2. Descarga de Datos

In [ ]:
def fetch_yfinance(tickers: dict, start: str, end: str) -> pd.DataFrame:
    """Descarga precios de cierre desde yfinance."""
    frames = {}
    for name, ticker in tickers.items():
        try:
            raw = yf.download(ticker, start=start, end=end, progress=False, auto_adjust=True)
            frames[name] = raw['Close'].squeeze()
            print(f'  ✓  {name:<8} ({ticker})  →  {len(raw)} obs')
        except Exception as exc:
            print(f'  ✗  {name:<8} ({ticker})  →  ERROR: {exc}')
    return pd.DataFrame(frames)


def fetch_stooq(tickers: dict, start: str, end: str) -> pd.DataFrame:
    """Descarga datos desde stooq via pandas_datareader."""
    frames = {}
    for name, ticker in tickers.items():
        try:
            raw = web.DataReader(ticker, 'stooq', start=start, end=end)
            raw = raw.sort_index()          # stooq devuelve orden descendente
            frames[name] = raw['Close']
            print(f'  ✓  {name:<8} ({ticker})  →  {len(raw)} obs')
        except Exception as exc:
            print(f'  ✗  {name:<8} ({ticker})  →  ERROR: {exc}')
            print(f'       Verifica el ticker en stooq.com o usa la alternativa FRED.')
    return pd.DataFrame(frames)


print('yfinance:')
df_yf = fetch_yfinance(YF_TICKERS, START, END)

print('\nstooq:')
df_stooq = fetch_stooq(STOOQ_TICKERS, START, END)

In [ ]:
# ─── Alternativa: si stooq no funciona, descomentar para usar FRED ─────────────
# FRED_TICKERS = {
#     'de2y': 'EUR3MTD156N',   # Euribor 3M (proxy tipos corto EUR)
#     'ch2y': 'CHF3MTD156N',   # CHF LIBOR 3M (proxy tipos corto CHF)
# }
# df_stooq = pd.DataFrame()
# for name, ticker in FRED_TICKERS.items():
#     s = web.DataReader(ticker, 'fred', start=START, end=END)[ticker]
#     df_stooq[name] = s
# df_stooq = df_stooq.resample('B').last()  # alinear a días hábiles

## 3. Preprocesado y Alineación

In [ ]:
# Combinar todos los datos
df_raw = df_yf.join(df_stooq, how='outer')

# Filtrar el periodo de interés y días hábiles
df_raw = df_raw.loc[START:END]

# Forward fill para festivos nacionales (máx. 3 días para evitar distorsiones)
df = df_raw.ffill(limit=3).dropna()

print(f'Observaciones finales: {len(df)}')
print(f'Rango: {df.index[0].date()} → {df.index[-1].date()}')
display(df.describe())

In [ ]:
# ─── Variables derivadas ───────────────────────────────────────────────────────
df['ird']       = df['de1y'] - df['ch1y']      # Diferencial de tipos (IRD)
df['log_eurchf']= np.log(df['eurchf'])          # Log del tipo de cambio
df['log_vix']   = np.log(df['vix'])             # Log del VIX

# Primeras diferencias (para ECM)
df['d_log_eurchf'] = df['log_eurchf'].diff()
df['d_ird']        = df['ird'].diff()
df['d_log_vix']    = df['log_vix'].diff()

df_model = df.dropna()
print(f'Observaciones para el modelo: {len(df_model)}')

## 4. Análisis Exploratorio

In [ ]:
fig = plt.figure(figsize=(14, 10))
gs  = gridspec.GridSpec(3, 2, figure=fig, hspace=0.45, wspace=0.35)

ax1 = fig.add_subplot(gs[0, :])
ax2 = fig.add_subplot(gs[1, 0])
ax3 = fig.add_subplot(gs[1, 1])
ax4 = fig.add_subplot(gs[2, 0])
ax5 = fig.add_subplot(gs[2, 1])

# EUR/CHF
ax1.plot(df.index, df['eurchf'], color='#2c6fad', lw=1.2)
ax1.set_title('EUR/CHF – Tipo de cambio diario', fontweight='bold')
ax1.set_ylabel('EUR/CHF')
ax1.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

# Bono alemán 1Y
ax2.plot(df.index, df['de1y'], color='#c0392b', lw=1)
ax2.set_title('Bono Alemán 1Y (%)')
ax2.set_ylabel('Rendimiento (%)')
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

# Bono suizo 1Y
ax3.plot(df.index, df['ch1y'], color='#e67e22', lw=1)
ax3.set_title('Bono Suizo 1Y (%)')
ax3.set_ylabel('Rendimiento (%)')
ax3.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

# Diferencial IRD
ax4.fill_between(df.index, df['ird'], 0,
                  where=df['ird'] >= 0, color='#27ae60', alpha=0.6, label='IRD > 0 (EUR ventaja)')
ax4.fill_between(df.index, df['ird'], 0,
                  where=df['ird'] < 0, color='#e74c3c', alpha=0.6, label='IRD < 0 (CHF ventaja)')
ax4.axhline(0, color='black', lw=0.8, ls='--')
ax4.set_title('Diferencial de Tipos DE1Y − CH1Y')
ax4.set_ylabel('Diferencial (pp)')
ax4.legend(fontsize=8)
ax4.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

# VIX
ax5.plot(df.index, df['vix'], color='#8e44ad', lw=0.9)
ax5.axhline(20, color='gray', lw=0.8, ls='--', label='VIX=20')
ax5.axhline(30, color='#e74c3c', lw=0.8, ls='--', label='VIX=30')
ax5.set_title('VIX – Proxy Safe Haven')
ax5.set_ylabel('Nivel')
ax5.legend(fontsize=8)
ax5.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

plt.suptitle('Variables del Modelo Fair Value EUR/CHF', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Matriz de correlaciones
corr_cols = ['log_eurchf', 'ird', 'log_vix']
corr_labels = ['log(EUR/CHF)', 'IRD (DE1Y−CH1Y)', 'log(VIX)']

corr = df_model[corr_cols].corr()
corr.index = corr.columns = corr_labels

fig, ax = plt.subplots(figsize=(5, 4))
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(corr, annot=True, fmt='.3f', cmap='RdBu_r', center=0,
            linewidths=0.5, ax=ax, mask=mask, square=True,
            cbar_kws={'shrink': 0.8})
ax.set_title('Correlaciones (niveles)', fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Tests Estadísticos

### 5.1 Test de Raíz Unitaria (ADF)

Si las series son I(1) (no estacionarias en niveles pero sí en diferencias), debemos verificar **cointegración** antes de hacer la regresión en niveles.

In [ ]:
def adf_test(series: pd.Series, name: str, lags: int = None) -> dict:
    """Realiza el test ADF y devuelve un resumen."""
    result = adfuller(series.dropna(), maxlag=lags, autolag='AIC', regression='c')
    return {
        'Variable'  : name,
        'ADF stat'  : round(result[0], 4),
        'p-value'   : round(result[1], 4),
        'Lags used' : result[2],
        '1%'        : round(result[4]['1%'], 3),
        '5%'        : round(result[4]['5%'], 3),
        'I(1)?'     : 'Sí (no estac.)' if result[1] > 0.05 else 'No (estac.)'
    }

adf_results = [
    adf_test(df_model['log_eurchf'],  'log(EUR/CHF)'),
    adf_test(df_model['ird'],         'IRD (DE1Y−CH1Y)'),
    adf_test(df_model['log_vix'],     'log(VIX)'),
    adf_test(df_model['d_log_eurchf'],'Δlog(EUR/CHF)'),
    adf_test(df_model['d_ird'],       'ΔIRD'),
    adf_test(df_model['d_log_vix'],   'Δlog(VIX)'),
]

adf_df = pd.DataFrame(adf_results).set_index('Variable')
display(adf_df)
print('\nH0 del ADF: la serie tiene raíz unitaria (es no estacionaria).')
print('Si p-value > 0.05 → no rechazamos H0 → I(1)')

### 5.2 Test de Cointegración (Engle-Granger)

Si log(EUR/CHF), IRD y log(VIX) son I(1), verificamos si están cointegradas → existencia de una relación de largo plazo estable.

In [ ]:
y = df_model['log_eurchf']
X = df_model[['ird', 'log_vix']]

# Engle-Granger (test bivariante iterado)
print('=== Test de Cointegración Engle-Granger ===')
for col, label in [('ird', 'IRD'), ('log_vix', 'log(VIX)')]:
    stat, pval, _ = coint(y, df_model[col])
    print(f'  log(EUR/CHF) ~ {label}:  stat={stat:.4f}  p-value={pval:.4f}  '
          f'  {"→ COINTEGRADAS ✓" if pval < 0.05 else "→ no cointegradas"}')

print()

# Regresión auxiliar para residuos de cointegración multivariante
X_const = sm.add_constant(X)
res_coint = sm.OLS(y, X_const).fit()
resid_coint = res_coint.resid
adf_resid = adfuller(resid_coint, autolag='AIC', regression='nc')  # sin constante en residuos
print(f'ADF sobre residuos de la regresión multivariante:')
print(f'  stat={adf_resid[0]:.4f}  p-value={adf_resid[1]:.4f}  '
      f'  {"→ residuos estacionarios → cointegración ✓" if adf_resid[1] < 0.10 else "→ residuos no estacionarios"}')

## 6. Modelo de Largo Plazo (Fair Value OLS)

In [ ]:
# Regresión de cointegración: log(EURCHF) = β0 + β1*IRD + β2*log(VIX) + ε
X_lr = sm.add_constant(df_model[['ird', 'log_vix']])
y_lr = df_model['log_eurchf']

ols_lr = sm.OLS(y_lr, X_lr).fit(cov_type='HAC', cov_kwds={'maxlags': 21})  # Newey-West
print(ols_lr.summary())

In [ ]:
# Diagnóstico de residuos del modelo de largo plazo
residuals_lr = ols_lr.resid

fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))

# Residuos en el tiempo
axes[0].plot(residuals_lr.index, residuals_lr, lw=0.7, color='#2c6fad')
axes[0].axhline(0, color='black', lw=0.8)
axes[0].set_title('Residuos en el tiempo')
axes[0].set_ylabel('Residuo')
axes[0].xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

# Histograma de residuos
axes[1].hist(residuals_lr, bins=60, color='#3498db', edgecolor='white', alpha=0.8)
axes[1].set_title('Distribución de residuos')
axes[1].set_xlabel('Residuo')

# ACF de residuos
from statsmodels.graphics.tsaplots import plot_acf
plot_acf(residuals_lr, lags=40, ax=axes[2], alpha=0.05)
axes[2].set_title('ACF de residuos')

plt.suptitle('Diagnóstico – Modelo de Largo Plazo', fontweight='bold')
plt.tight_layout()
plt.show()

lb = acorr_ljungbox(residuals_lr, lags=[10, 20], return_df=True)
print('Test Ljung-Box (H0: no autocorrelación):')
display(lb)

## 7. Modelo de Corrección de Error (ECM)

Captura la dinámica de **corto plazo** y la velocidad de retorno al fair value:

$$\Delta\log(\text{EURCHF})_t = \alpha \cdot \hat{\varepsilon}_{t-1} + \gamma_1 \Delta\text{IRD}_t + \gamma_2 \Delta\log(\text{VIX})_t + u_t$$

donde $\hat{\varepsilon}_{t-1}$ es el residuo rezagado del modelo de largo plazo (término de corrección de error). **Se espera $\alpha < 0$** (reversión a la media).

In [ ]:
# Término de corrección de error (ECT): residuo del modelo de largo plazo, rezagado 1 período
df_model = df_model.copy()
df_model['ect_lag1'] = residuals_lr.shift(1)

df_ecm = df_model.dropna(subset=['d_log_eurchf', 'd_ird', 'd_log_vix', 'ect_lag1'])

X_ecm = sm.add_constant(df_ecm[['ect_lag1', 'd_ird', 'd_log_vix']])
y_ecm = df_ecm['d_log_eurchf']

ols_ecm = sm.OLS(y_ecm, X_ecm).fit(cov_type='HAC', cov_kwds={'maxlags': 5})
print(ols_ecm.summary())

alpha = ols_ecm.params['ect_lag1']
if alpha < 0:
    dias_retorno = -1 / alpha
    print(f'\n→ α = {alpha:.4f} < 0 ✓  →  velocidad de retorno al fair value: ~{dias_retorno:.1f} días hábiles')
else:
    print(f'\n→ α = {alpha:.4f} ≥ 0  →  no se observa reversión significativa en este período')

## 8. Fair Value y Señales

In [ ]:
# Fair value en niveles
df_model['fair_value_log'] = ols_lr.fittedvalues
df_model['fair_value']     = np.exp(df_model['fair_value_log'])

# Desvío: (precio actual - fair value) / fair value * 100
df_model['desvio_pct'] = (df_model['eurchf'] - df_model['fair_value']) / df_model['fair_value'] * 100

# Bandas de señal basadas en desviaciones típicas históricas
sigma      = df_model['desvio_pct'].std()
mean_desv  = df_model['desvio_pct'].mean()

print(f'Desviación típica del desvío: {sigma:.2f}%')
print(f'Media del desvío:             {mean_desv:.2f}%')
print(f'\nÚltimo valor:')
last = df_model.iloc[-1]
print(f'  Fecha        : {df_model.index[-1].date()}')
print(f'  EUR/CHF obs. : {last["eurchf"]:.4f}')
print(f'  Fair Value   : {last["fair_value"]:.4f}')
print(f'  Desvío       : {last["desvio_pct"]:+.2f}%  '
      f'({last["desvio_pct"]/sigma:+.2f}σ)')

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 12), sharex=True,
                         gridspec_kw={'height_ratios': [3, 2, 2], 'hspace': 0.15})

# ── Panel 1: EUR/CHF observado vs Fair Value ──────────────────────────────────
ax = axes[0]
ax.plot(df_model.index, df_model['eurchf'],     lw=1.2, color='#2c6fad', label='EUR/CHF observado', zorder=3)
ax.plot(df_model.index, df_model['fair_value'], lw=1.5, color='#e74c3c',
        ls='--', label='Fair Value (modelo)', zorder=3)
ax.fill_between(df_model.index,
                df_model['eurchf'], df_model['fair_value'],
                where=df_model['eurchf'] > df_model['fair_value'],
                color='#e74c3c', alpha=0.15, label='Sobrevalorado')
ax.fill_between(df_model.index,
                df_model['eurchf'], df_model['fair_value'],
                where=df_model['eurchf'] < df_model['fair_value'],
                color='#27ae60', alpha=0.15, label='Infravalorado')
ax.set_ylabel('EUR/CHF')
ax.set_title('EUR/CHF: Precio Observado vs Fair Value', fontweight='bold', fontsize=12)
ax.legend(loc='upper right', fontsize=9)

# ── Panel 2: Desvío en % ──────────────────────────────────────────────────────
ax2 = axes[1]
ax2.fill_between(df_model.index, df_model['desvio_pct'], 0,
                 where=df_model['desvio_pct'] > 0, color='#e74c3c', alpha=0.5)
ax2.fill_between(df_model.index, df_model['desvio_pct'], 0,
                 where=df_model['desvio_pct'] <= 0, color='#27ae60', alpha=0.5)

# Bandas ±1σ y ±2σ
for n, ls, col in [(1, '--', '#e67e22'), (2, ':', '#c0392b')]:
    ax2.axhline( n * sigma + mean_desv, color=col, lw=1, ls=ls, label=f'+{n}σ')
    ax2.axhline(-n * sigma + mean_desv, color='#27ae60', lw=1, ls=ls, label=f'−{n}σ')
ax2.axhline(mean_desv, color='black', lw=0.8, ls='-')
ax2.set_ylabel('Desvío (%)')
ax2.set_title('Desvío del Fair Value (%)', fontweight='bold')
ax2.legend(loc='upper right', fontsize=8, ncol=4)

# ── Panel 3: Desvío en σ ──────────────────────────────────────────────────────
ax3 = axes[2]
zscore = (df_model['desvio_pct'] - mean_desv) / sigma
ax3.plot(df_model.index, zscore, lw=0.8, color='#8e44ad')
ax3.axhline(0,  color='black', lw=0.8)
ax3.axhline( 1, color='#e67e22', lw=1, ls='--')
ax3.axhline(-1, color='#e67e22', lw=1, ls='--')
ax3.axhline( 2, color='#c0392b', lw=1, ls=':')
ax3.axhline(-2, color='#c0392b', lw=1, ls=':')

# Marcar episodios > 2σ
extreme = zscore.abs() > 2
ax3.fill_between(df_model.index, zscore, 0, where=(zscore > 2),  color='#c0392b', alpha=0.4)
ax3.fill_between(df_model.index, zscore, 0, where=(zscore < -2), color='#27ae60', alpha=0.4)
ax3.set_ylabel('Z-score (σ)')
ax3.set_title('Desvío Normalizado (Z-score)', fontweight='bold')
ax3.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax3.xaxis.set_major_locator(mdates.YearLocator())

plt.suptitle('Modelo Fair Value EUR/CHF – Resultados', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ─── Señales de trading / posicionamiento ─────────────────────────────────────
def get_signal(z: float) -> str:
    if z > 2:    return '🔴 VENDER EUR/CHF  (muy sobrevalorado, >+2σ)'
    elif z > 1:  return '🟠 Sesgo vendedor   (+1σ a +2σ)'
    elif z < -2: return '🟢 COMPRAR EUR/CHF  (muy infravalorado, <−2σ)'
    elif z < -1: return '🔵 Sesgo comprador   (−2σ a −1σ)'
    else:        return '⚪ NEUTRAL            (dentro de ±1σ)'

last_z = zscore.iloc[-1]
last_date = df_model.index[-1]

print('═' * 55)
print(f'  POSICIONAMIENTO ACTUAL – {last_date.date()}')
print('═' * 55)
print(f'  EUR/CHF spot   : {last["eurchf"]:.4f}')
print(f'  Fair Value     : {last["fair_value"]:.4f}')
print(f'  Desvío         : {last["desvio_pct"]:+.2f}%   ({last_z:+.2f}σ)')
print(f'  Señal          : {get_signal(last_z)}')
print(f'  IRD (DE1Y−CH1Y): {last["ird"]:+.2f} pp')
print(f'  VIX            : {last["vix"]:.1f}')
print('═' * 55)

# Tabla de episodios extremos (>2σ) históricos
extremos = df_model[zscore.abs() > 2][['eurchf', 'fair_value', 'desvio_pct', 'ird', 'vix']].copy()
extremos.index = extremos.index.date
extremos.columns = ['EUR/CHF', 'Fair Value', 'Desvío %', 'IRD', 'VIX']
print(f'\nEpisodios con desvío > |2σ| ({len(extremos)} días):')
display(extremos.head(20))

## 9. Resumen del Modelo

| Componente | Descripción |
|---|---|
| **Modelo LR** | OLS: log(EUR/CHF) ~ IRD + log(VIX) con errores HAC (Newey-West) |
| **Modelo CP** | ECM: Δlog(EUR/CHF) ~ ECT(t-1) + ΔIRD + Δlog(VIX) |
| **Fair Value** | Valor ajustado del modelo de largo plazo |
| **Señal +2σ** | EUR/CHF muy por encima del fair value → sesgo a la baja |
| **Señal −2σ** | EUR/CHF muy por debajo del fair value → sesgo al alza |

### Limitaciones
- El modelo no captura intervenciones del SNB ni cambios estructurales de política monetaria.
- Los rendimientos de bonos pueden estar distorsionados en periodos de QE/QT de BCE y SNB.
- El VIX es un proxy imperfecto del safe-haven; alternativas: VSTOXX (`^V2TX`), spreads de crédito, precio del oro.
- No es un modelo predictivo sino descriptivo del equilibrio contemporáneo.